# Домашнее задание: Латентная диффузия (Latent Diffusion Model)

## Генерация изображений 128×128 через DDPM в латентном пространстве VAE

Это продолжение предыдущего задания. Мы используем обученный VAE (128×128×3 → 16×16×4) и обучаем диффузионную модель в его латентном пространстве.

В ноутбуке все клетки должны выполняться без ошибок при последовательном их выполнении.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import math
import random
from typing import Tuple, List, Optional

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib_inline

import tqdm.autonotebook as tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid

%matplotlib inline
matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg')

torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# Загрузка обученного VAE и подготовка латентов

Загружаем VAE из предыдущего задания и прогоняем весь датасет через encoder, чтобы получить датасет латентов $z \in \mathbb{R}^{4 \times 16 \times 16}$.

Диффузия будет обучаться **только на латентах** — VAE больше не обучается.

## Копируем архитектуру VAE

Нам нужны те же классы `ResBlock`, `Encoder`, `Decoder`, `VAE` из предыдущего задания. Они определены здесь для загрузки чекпоинта.

In [ ]:
class ResBlock(nn.Module):
    """Residual block with GroupNorm and SiLU (from VAE homework)."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(32, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(32, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Downsample(nn.Module):
    def __init__(self, ch): super().__init__(); self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, ch): super().__init__(); self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x): return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))


class Encoder(nn.Module):
    def __init__(self, latent_channels=4, ch=64, ch_mults=(1, 2, 4)):
        super().__init__()
        channels = [ch * m for m in ch_mults]
        self.conv_in = nn.Conv2d(3, ch, 3, padding=1)
        self.down_blocks = nn.ModuleList()
        in_ch = ch
        for out_ch in channels:
            self.down_blocks.append(nn.ModuleList([ResBlock(in_ch, out_ch), ResBlock(out_ch, out_ch), Downsample(out_ch)]))
            in_ch = out_ch
        self.mid_block1 = ResBlock(channels[-1], channels[-1])
        self.mid_block2 = ResBlock(channels[-1], channels[-1])
        self.norm_out = nn.GroupNorm(32, channels[-1])
        self.conv_out = nn.Conv2d(channels[-1], 2 * latent_channels, 3, padding=1)

    def forward(self, x):
        h = self.conv_in(x)
        for res1, res2, down in self.down_blocks:
            h = down(res2(res1(h)))
        h = self.mid_block2(self.mid_block1(h))
        return self.conv_out(F.silu(self.norm_out(h)))


class Decoder(nn.Module):
    def __init__(self, latent_channels=4, ch=64, ch_mults=(1, 2, 4)):
        super().__init__()
        channels = [ch * m for m in ch_mults]
        self.conv_in = nn.Conv2d(latent_channels, channels[-1], 3, padding=1)
        self.mid_block1 = ResBlock(channels[-1], channels[-1])
        self.mid_block2 = ResBlock(channels[-1], channels[-1])
        self.up_blocks = nn.ModuleList()
        in_ch = channels[-1]
        for out_ch in reversed(channels):
            self.up_blocks.append(nn.ModuleList([ResBlock(in_ch, out_ch), ResBlock(out_ch, out_ch), Upsample(out_ch)]))
            in_ch = out_ch
        self.norm_out = nn.GroupNorm(32, channels[0])
        self.conv_out = nn.Conv2d(channels[0], 3, 3, padding=1)

    def forward(self, z):
        h = self.mid_block2(self.mid_block1(self.conv_in(z)))
        for res1, res2, up in self.up_blocks:
            h = up(res2(res1(h)))
        return self.conv_out(F.silu(self.norm_out(h)))


class VAE(nn.Module):
    def __init__(self, latent_channels=4, ch=64, ch_mults=(1, 2, 4)):
        super().__init__()
        self.latent_channels = latent_channels
        self.encoder = Encoder(latent_channels, ch, ch_mults)
        self.decoder = Decoder(latent_channels, ch, ch_mults)

    def encode(self, x):
        h = self.encoder(x)
        mu, logvar = torch.chunk(h, 2, dim=1)
        return mu, logvar

    def decode(self, z):
        return self.decoder(z)

## Загрузка чекпоинта и подготовка латентного датасета

In [ ]:
# Загружаем обученный VAE
checkpoint = torch.load('vae_for_ldm.pt', map_location=device)
vae = VAE(**{k: v for k, v in checkpoint['config'].items() if k in ('latent_channels', 'ch', 'ch_mults')})
vae.load_state_dict(checkpoint['vae_state_dict'])
vae = vae.to(device).eval()
for p in vae.parameters():
    p.requires_grad_(False)
print(f'VAE loaded. Latent space: {checkpoint["config"]["img_size"]}x{checkpoint["config"]["img_size"]}x3 -> '
      f'{checkpoint["config"]["latent_size"]}x{checkpoint["config"]["latent_size"]}x{checkpoint["config"]["latent_channels"]}')

In [ ]:
import tarfile

class ImageNetteDataset(Dataset):
    """ImageNette dataset (same as VAE homework)."""
    def __init__(self, root, split='train', img_size=128):
        self.transform = transforms.Compose([
            transforms.Resize(img_size), transforms.CenterCrop(img_size),
            transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        split_dir = os.path.join(root, split)
        self.image_paths = []
        for cd in sorted(os.listdir(split_dir)):
            cp = os.path.join(split_dir, cd)
            if not os.path.isdir(cp): continue
            for f in sorted(os.listdir(cp)):
                if f.lower().endswith(('.png','.jpg','.jpeg')):
                    self.image_paths.append(os.path.join(cp, f))

    def __len__(self): return len(self.image_paths)
    def __getitem__(self, i):
        return self.transform(Image.open(self.image_paths[i]).convert('RGB'))

if not os.path.exists('./imagenette2-160'):
    if not os.path.exists('./imagenette2-160.tgz'):
        !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz
    with tarfile.open('./imagenette2-160.tgz', 'r:gz') as tar:
        tar.extractall('.')

train_dataset = ImageNetteDataset('./imagenette2-160', 'train', 128)
val_dataset = ImageNetteDataset('./imagenette2-160', 'val', 128)
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
@torch.inference_mode()
def encode_dataset(vae, dataset, device, batch_size=64):
    """Encode entire dataset to latents using pretrained VAE.

    Args:
        vae: Pretrained VAE model (frozen).
        dataset: Image dataset.
        device: Compute device.
        batch_size: Encoding batch size.

    Returns:
        Tensor of latent codes (N, 4, 16, 16).
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    all_latents = []
    for batch in tqdm.tqdm(loader, desc='Encoding'):
        images = batch.to(device)
        mu, _ = vae.encode(images)
        all_latents.append(mu.cpu())
    return torch.cat(all_latents, dim=0)

# Кэшируем латенты на диск для быстрого перезапуска
if os.path.exists('latents_train.pt'):
    train_latents = torch.load('latents_train.pt')
    val_latents = torch.load('latents_val.pt')
    print('Loaded cached latents')
else:
    train_latents = encode_dataset(vae, train_dataset, device)
    val_latents = encode_dataset(vae, val_dataset, device)
    torch.save(train_latents, 'latents_train.pt')
    torch.save(val_latents, 'latents_val.pt')
    print('Encoded and cached latents')

print(f'Train latents: {train_latents.shape}, range: [{train_latents.min():.2f}, {train_latents.max():.2f}]')
print(f'Val latents:   {val_latents.shape}')

### Нормализация латентов

В Stable Diffusion латенты нормализуются делением на их стандартное отклонение, чтобы привести дисперсию к ~1. Это важно для совместимости с noise schedule диффузии.

In [ ]:
# Вычисляем и применяем scaling factor
latent_std = train_latents.std()
scaling_factor = 1.0 / latent_std
print(f'Latent std: {latent_std:.4f}, scaling factor: {scaling_factor:.4f}')

train_latents_scaled = train_latents * scaling_factor
val_latents_scaled = val_latents * scaling_factor

print(f'Scaled train latents std: {train_latents_scaled.std():.4f}')

train_latent_loader = DataLoader(
    TensorDataset(train_latents_scaled),
    batch_size=64, shuffle=True, drop_last=True
)

In [ ]:
# Визуализация: декодируем несколько латентов обратно
@torch.inference_mode()
def decode_and_show(vae, latents_scaled, scaling_factor, n=8, title=''):
    latents = latents_scaled[:n].to(device) / scaling_factor
    images = vae.decode(latents).clamp(-1, 1)
    grid = make_grid(images, nrow=n, normalize=True, value_range=(-1, 1))
    fig, ax = plt.subplots(1, 1, figsize=(n * 2, 2.5))
    ax.imshow(grid.permute(1, 2, 0).cpu().numpy())
    ax.set_title(title or 'Decoded from latents')
    ax.axis('off')
    plt.tight_layout(); plt.show()

decode_and_show(vae, train_latents_scaled, scaling_factor, title='Train images decoded from cached latents')

# Noise Schedule и Forward Process

Используем стандартное VP (Variance-Preserving) линейное расписание:

$$q(z_t \mid z_0) = \mathcal{N}(z_t; \sqrt{\bar\alpha_t}\, z_0, (1 - \bar\alpha_t)\, I)$$

где $\bar\alpha_t = \prod_{s=1}^{t} (1 - \beta_s)$, $\beta_t$ линейно растёт от $\beta_{\text{start}}$ до $\beta_{\text{end}}$.

Реализуйте класс `NoiseSchedule`, который предвычисляет все необходимые тензоры:
- `betas`: $\beta_t$ — линейное расписание от `beta_start` до `beta_end`, $T$ значений
- `alphas`: $\alpha_t = 1 - \beta_t$
- `alpha_bar`: $\bar\alpha_t = \prod_{s=1}^{t} \alpha_s$ (кумулятивное произведение)
- `alpha_bar_prev`: $\bar\alpha_{t-1}$ (сдвинутое на 1, $\bar\alpha_0 = 1$)
- `posterior_variance`: $\tilde\beta_t = \beta_t \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}$

А также метод `q_sample`, который по чистому $z_0$ и шагу $t$ возвращает зашумлённый $z_t$:
$$z_t = \sqrt{\bar\alpha_t}\, z_0 + \sqrt{1 - \bar\alpha_t}\, \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

In [ ]:
class NoiseSchedule:
    """VP (Variance-Preserving) linear noise schedule for DDPM.

    Precomputes all schedule tensors: betas, alphas, alpha_bar,
    and posterior variance for efficient training and sampling.

    Args:
        T: Number of diffusion timesteps.
        beta_start: Starting noise level.
        beta_end: Ending noise level.
        device: Compute device.
    """

    def __init__(self, T: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02,
                 device: str = 'cpu'):
        self.T = T
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.alpha_bar_prev = F.pad(self.alpha_bar[:-1], (1, 0), value=1.0)
        self.posterior_variance = (
            self.betas * (1.0 - self.alpha_bar_prev) / (1.0 - self.alpha_bar)
        )

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor,
                 noise: Optional[torch.Tensor] = None) -> tuple:
        """Forward process: sample z_t from q(z_t | x_0).

        Args:
            x0: Clean data of shape (B, C, H, W).
            t: Timestep indices of shape (B,).
            noise: Optional pre-sampled noise.

        Returns:
            Tuple of (z_t, noise).
        """
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab = self.alpha_bar[t].sqrt()[:, None, None, None]
        sqrt_1m = (1.0 - self.alpha_bar[t]).sqrt()[:, None, None, None]
        return sqrt_ab * x0 + sqrt_1m * noise, noise

In [ ]:
# Тесты для NoiseSchedule
schedule = NoiseSchedule(T=1000, device=device)
print(f'Schedule: T={schedule.T}, beta=[{schedule.betas[0]:.5f}, {schedule.betas[-1]:.4f}]')

assert schedule.betas.shape == (1000,)
assert schedule.alpha_bar.shape == (1000,)
assert schedule.alpha_bar[0] > 0.99, 'alpha_bar[0] should be close to 1'
assert schedule.alpha_bar[-1] < 0.05, 'alpha_bar[-1] should be close to 0'
assert abs(schedule.alpha_bar_prev[0].item() - 1.0) < 1e-6

# Test q_sample
z0 = torch.randn(4, 4, 16, 16, device=device)
t = torch.tensor([0, 100, 500, 999], device=device)
zt, eps = schedule.q_sample(z0, t)
assert zt.shape == z0.shape
# At t=0, z_t should be very close to z0
assert (zt[0] - z0[0]).abs().max() < 0.02, 'At t=0, z_t should be ~z0'

print('All NoiseSchedule tests passed!')

In [ ]:
# Визуализация forward процесса в латентном пространстве
z0 = train_latents_scaled[0:1].to(device)

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
t_values = [0, 50, 200, 500, 800, 999]

for ax, t_val in zip(axes, t_values):
    t = torch.tensor([t_val], device=device)
    zt, _ = schedule.q_sample(z0, t)
    # Decode to image
    with torch.inference_mode():
        img = vae.decode(zt / scaling_factor).clamp(-1, 1)
    img_np = img[0].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax.imshow(np.clip(img_np, 0, 1))
    ax.set_title(f't = {t_val}')
    ax.axis('off')

fig.suptitle('Forward diffusion in latent space (decoded to pixels)', fontsize=14)
plt.tight_layout(); plt.show()

# U-Net для латентного пространства

Адаптируем U-Net из лекции под латентное пространство $16 \times 16 \times 4$:

- **Вход/выход**: 4 канала (вместо 1 для MNIST)
- **Разрешения**: $16 \to 8 \to 4$ (encoder), $4 \to 8 \to 16$ (decoder)
- **Time conditioning**: sinusoidal embedding + MLP → добавляется в каждый ResBlock
- **Base channels**: 128 (больше чем для MNIST, т.к. задача сложнее)

## Sinusoidal Positional Embedding

Временной шаг $t$ кодируется синусоидальным позиционным эмбеддингом (как в Transformer / DDPM):

$$\text{emb}_{2i}(t) = \sin\left(\frac{t}{10000^{2i/d}}\right), \quad \text{emb}_{2i+1}(t) = \cos\left(\frac{t}{10000^{2i/d}}\right)$$

In [ ]:
class SinusoidalPosEmb(nn.Module):
    """Sinusoidal positional embedding for timestep conditioning.

    Args:
        dim: Embedding dimension.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """Compute sinusoidal embedding.

        Args:
            t: Timestep indices of shape (B,).

        Returns:
            Embeddings of shape (B, dim).
        """
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

In [ ]:
# Тест SinusoidalPosEmb
emb = SinusoidalPosEmb(128)
t = torch.tensor([0, 100, 500, 999])
out = emb(t)
assert out.shape == (4, 128), f'Expected (4, 128), got {out.shape}'

# Разные t должны давать разные эмбеддинги
assert not torch.allclose(out[0], out[1])
print('SinusoidalPosEmb test passed!')

## Residual Block с time conditioning

Residual block для диффузионного U-Net. Отличие от ResBlock в VAE — **time conditioning**: эмбеддинг времени проецируется линейным слоем и добавляется к feature map между двумя свёртками.

```
x --> [GN(8) -> SiLU -> Conv3x3] --> (+) --> [GN(8) -> SiLU -> Conv3x3] --> (+) --> out
                                      ^                                       ^
                              [SiLU -> Linear(t_emb)]                    skip(x)
```

**Важно:** GroupNorm с `num_groups=8` (а не 32 как в VAE, т.к. каналов может быть 128).

In [ ]:
class DiffResBlock(nn.Module):
    """Residual block with time conditioning for diffusion U-Net.

    Args:
        in_ch: Input channels.
        out_ch: Output channels.
        time_dim: Time embedding dimension.
    """

    def __init__(self, in_ch: int, out_ch: int, time_dim: int):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.GroupNorm(8, in_ch), nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        )
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.conv2 = nn.Sequential(
            nn.GroupNorm(8, out_ch), nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """Forward pass with time conditioning.

        Args:
            x: Input of shape (B, C_in, H, W).
            t_emb: Time embedding of shape (B, time_dim).

        Returns:
            Output of shape (B, C_out, H, W).
        """
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.shortcut(x)

In [ ]:
# Тесты для DiffResBlock
block = DiffResBlock(128, 128, 256)
x = torch.randn(2, 128, 16, 16)
t_emb = torch.randn(2, 256)
out = block(x, t_emb)
assert out.shape == (2, 128, 16, 16)

block2 = DiffResBlock(128, 256, 256)
out2 = block2(x, t_emb)
assert out2.shape == (2, 256, 16, 16)

print('DiffResBlock tests passed!')

## Latent U-Net

U-Net для денойзинга в латентном пространстве $16 \times 16 \times 4$.

Архитектура:
```
Input (4, 16, 16) -> [Conv3x3] -> (C, 16, 16)
  Encoder:
    enc1: DiffResBlock(C, C)        -> (C, 16, 16)
    down1: Conv3x3 stride=2         -> (C, 8, 8)
    enc2: DiffResBlock(C, 2C)       -> (2C, 8, 8)
    down2: Conv3x3 stride=2         -> (2C, 4, 4)
  Bottleneck:
    DiffResBlock(2C, 2C)            -> (2C, 4, 4)
  Decoder:
    up2: ConvTranspose 4x4 stride=2 -> (2C, 8, 8)
    dec2: DiffResBlock(4C, C)       -> (C, 8, 8)     # 4C из-за skip connection
    up1: ConvTranspose 4x4 stride=2 -> (C, 16, 16)
    dec1: DiffResBlock(2C, C)       -> (C, 16, 16)    # 2C из-за skip connection
  Output: [GN(8) -> SiLU -> Conv1x1] -> (4, 16, 16)
```

где $C = 128$ (base_channels), time_dim $= 256$.

**Time MLP**: `SinusoidalPosEmb(time_dim) -> Linear(time_dim, time_dim*2) -> SiLU -> Linear(time_dim*2, time_dim)`

**Важно:** В decoder'е после каждого upsample перед DiffResBlock конкатенируем skip connection из encoder'а (по каналам). Поэтому `dec2` принимает $4C$ каналов ($2C$ от upsample + $2C$ skip), а `dec1` принимает $2C$ ($C$ + $C$ skip).

In [ ]:
class LatentUNet(nn.Module):
    """U-Net for denoising in 16x16x4 latent space.

    Architecture: 16x16 -> 8x8 -> 4x4 (bottleneck) -> 8x8 -> 16x16.
    Uses skip connections, time conditioning, and residual blocks.

    Args:
        in_channels: Latent channels (4 for SD-style VAE).
        base_channels: Base feature map count.
        time_dim: Dimension of time embedding.
    """

    def __init__(self, in_channels: int = 4, base_channels: int = 128, time_dim: int = 256):
        super().__init__()
        C = base_channels

        # Time embedding MLP
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_dim),
            nn.Linear(time_dim, time_dim * 2), nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim),
        )

        # Input projection
        self.input_proj = nn.Conv2d(in_channels, C, 3, padding=1)

        # Encoder: 16 -> 8 -> 4
        self.enc1 = DiffResBlock(C, C, time_dim)
        self.down1 = nn.Conv2d(C, C, 3, stride=2, padding=1)       # 16->8
        self.enc2 = DiffResBlock(C, C * 2, time_dim)
        self.down2 = nn.Conv2d(C * 2, C * 2, 3, stride=2, padding=1)  # 8->4

        # Bottleneck (4x4)
        self.bottleneck = DiffResBlock(C * 2, C * 2, time_dim)

        # Decoder: 4 -> 8 -> 16
        self.up2 = nn.ConvTranspose2d(C * 2, C * 2, 4, stride=2, padding=1)  # 4->8
        self.dec2 = DiffResBlock(C * 4, C, time_dim)  # concat skip
        self.up1 = nn.ConvTranspose2d(C, C, 4, stride=2, padding=1)          # 8->16
        self.dec1 = DiffResBlock(C * 2, C, time_dim)  # concat skip

        # Output
        self.out = nn.Sequential(
            nn.GroupNorm(8, C), nn.SiLU(),
            nn.Conv2d(C, in_channels, 1),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict noise eps_theta(z_t, t).

        Args:
            x: Noisy latent of shape (B, 4, 16, 16).
            t: Timestep indices of shape (B,).

        Returns:
            Predicted noise of shape (B, 4, 16, 16).
        """
        t_emb = self.time_mlp(t)
        x = self.input_proj(x)

        h1 = self.enc1(x, t_emb)                       # (B, C, 16, 16)
        h2 = self.enc2(self.down1(h1), t_emb)           # (B, 2C, 8, 8)
        h = self.bottleneck(self.down2(h2), t_emb)      # (B, 2C, 4, 4)

        h = self.up2(h)                                 # (B, 2C, 8, 8)
        h = self.dec2(torch.cat([h, h2], 1), t_emb)     # (B, C, 8, 8)
        h = self.up1(h)                                 # (B, C, 16, 16)
        h = self.dec1(torch.cat([h, h1], 1), t_emb)     # (B, C, 16, 16)

        return self.out(h)

In [ ]:
# Тесты для LatentUNet
model = LatentUNet(in_channels=4, base_channels=128, time_dim=256).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'LatentUNet parameters: {n_params:,}')

# Shape check
x_test = torch.randn(2, 4, 16, 16, device=device)
t_test = torch.randint(0, 1000, (2,), device=device)
out_test = model(x_test, t_test)
assert out_test.shape == (2, 4, 16, 16), f'Expected (2, 4, 16, 16), got {out_test.shape}'

print('LatentUNet shape check passed!')

# Обучение DDPM в латентном пространстве (3 балла)

Алгоритм обучения идентичен обычному DDPM, но работает на латентах:

1. $z_0$ — латент из датасета (уже закодирован VAE)
2. $t \sim \text{Uniform}\{1, \ldots, T\}$
3. $\varepsilon \sim \mathcal{N}(0, I)$
4. $z_t = \sqrt{\bar\alpha_t}\, z_0 + \sqrt{1 - \bar\alpha_t}\, \varepsilon$  (forward process)
5. Оптимизируем $\| \varepsilon - \varepsilon_\theta(z_t, t) \|^2$  (MSE между реальным и предсказанным шумом)

Реализуйте цикл обучения. Используйте:
- Optimizer: `AdamW` с `lr=2e-4` и `weight_decay=1e-4`
- Scheduler: `CosineAnnealingLR`
- Gradient clipping: `clip_grad_norm_(model.parameters(), 1.0)`
- Epochs: 60

In [ ]:
def train_latent_ddpm(model, schedule, train_loader, num_epochs=60, lr=2e-4):
    """Train DDPM noise predictor on precomputed latents.

    Args:
        model: U-Net noise prediction network.
        schedule: NoiseSchedule instance.
        train_loader: DataLoader of (latent,) tuples.
        num_epochs: Number of training epochs.
        lr: Learning rate.

    Returns:
        List of per-epoch average losses.
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    losses = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        pbar = tqdm.tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

        for (z0,) in pbar:
            z0 = z0.to(device)
            B = z0.shape[0]

            t = torch.randint(0, schedule.T, (B,), device=device)
            noise = torch.randn_like(z0)
            z_t, _ = schedule.q_sample(z0, t, noise)

            noise_pred = model(z_t, t)
            loss = F.mse_loss(noise_pred, noise)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        scheduler.step()
        avg = epoch_loss / n_batches
        losses.append(avg)
        print(f'  Epoch {epoch+1}: avg loss = {avg:.5f}')

    return losses

In [ ]:
set_global_seed(42)
losses = train_latent_ddpm(model, schedule, train_latent_loader, num_epochs=60, lr=2e-4)

# Save checkpoint
os.makedirs('checkpoints', exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'losses': losses,
    'scaling_factor': scaling_factor.item() if torch.is_tensor(scaling_factor) else scaling_factor,
}, 'checkpoints/latent_ddpm.pt')
print('Checkpoint saved!')

In [ ]:
# Кривая обучения
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, 'C0-', lw=2, marker='o', markersize=3)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Latent DDPM Training Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

#  Сэмплинг

##  DDPM Sampling

Стандартный DDPM сэмплинг: $z_T \sim \mathcal{N}(0, I)$, затем итеративно $z_{t-1} \sim p_\theta(z_{t-1} | z_t)$ для $t = T, \ldots, 1$.

Формула одного шага:

$$\mu_\theta(z_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( z_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}} \varepsilon_\theta(z_t, t) \right)$$

$$z_{t-1} = \mu_\theta(z_t, t) + \tilde\sigma_t \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I) \quad \text{(при } t > 0\text{)}$$

где $\tilde\sigma_t = \sqrt{\tilde\beta_t}$ — posterior variance.

После получения $z_0$ в латентном пространстве декодируем его VAE: $\hat{x} = \text{Decoder}(z_0 / \text{scaling\_factor})$.

In [ ]:
@torch.inference_mode()
def sample_ddpm_latent(model, schedule, vae, scaling_factor, n_samples=16,
                       return_intermediates=False):
    """DDPM sampling in latent space + VAE decode to pixels.

    Args:
        model: Trained noise prediction U-Net.
        schedule: NoiseSchedule instance.
        vae: Pretrained VAE decoder.
        scaling_factor: Latent normalization factor.
        n_samples: Number of images to generate.
        return_intermediates: Whether to return decoded intermediates.

    Returns:
        Generated images (B, 3, 128, 128) in [-1, 1].
        Optionally, list of intermediate decoded images.
    """
    model.eval()
    shape = (n_samples, 4, 16, 16)
    z = torch.randn(shape, device=device)
    intermediates = []

    for t in tqdm.tqdm(reversed(range(schedule.T)), total=schedule.T, desc='DDPM Sampling'):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps_pred = model(z, t_batch)

        alpha_t = schedule.alphas[t]
        alpha_bar_t = schedule.alpha_bar[t]
        beta_t = schedule.betas[t]

        mean = (1.0 / alpha_t.sqrt()) * (z - (beta_t / (1 - alpha_bar_t).sqrt()) * eps_pred)

        if t > 0:
            sigma = schedule.posterior_variance[t].sqrt()
            z = mean + sigma * torch.randn_like(z)
        else:
            z = mean

        if return_intermediates and t % 200 == 0:
            decoded = vae.decode(z / scaling_factor).clamp(-1, 1).cpu()
            intermediates.append(decoded)

    # Decode final latent to pixels
    images = vae.decode(z / scaling_factor).clamp(-1, 1)

    if return_intermediates:
        return images.cpu(), intermediates
    return images.cpu()

In [ ]:
# Генерация
set_global_seed(123)
samples = sample_ddpm_latent(model, schedule, vae, scaling_factor, n_samples=16)

grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title('DDPM Latent Diffusion Samples (128x128)', fontsize=14)
ax.axis('off')
plt.tight_layout(); plt.show()

## DDIM Sampling

DDIM позволяет генерировать за произвольное число шагов $S \ll T$. Это критически важно для практического использования — вместо 1000 шагов можно использовать 50.

Формула DDIM (детерминистический):

$$x_0^{\text{pred}} = \frac{z_t - \sqrt{1 - \bar\alpha_t} \cdot \varepsilon_\theta(z_t, t)}{\sqrt{\bar\alpha_t}}$$

$$z_{t_{\text{prev}}} = \sqrt{\bar\alpha_{t_{\text{prev}}}} \cdot x_0^{\text{pred}} + \sqrt{1 - \bar\alpha_{t_{\text{prev}}}} \cdot \varepsilon_\theta(z_t, t)$$

Подвыборка шагов: равномерно выбираем $S$ индексов из $\{0, \ldots, T-1\}$, идём по ним от большего к меньшему.

In [ ]:
@torch.inference_mode()
def sample_ddim_latent(model, schedule, vae, scaling_factor, n_samples=16, n_steps=50):
    """DDIM deterministic sampling in latent space + VAE decode.

    Args:
        model: Trained noise prediction U-Net.
        schedule: NoiseSchedule instance.
        vae: Pretrained VAE decoder.
        scaling_factor: Latent normalization factor.
        n_samples: Number of images to generate.
        n_steps: Number of DDIM steps (can be << T).

    Returns:
        Generated images (B, 3, 128, 128) in [-1, 1].
    """
    model.eval()

    step_indices = list(np.linspace(0, schedule.T - 1, n_steps, dtype=int))
    step_indices = sorted(set(step_indices), reverse=True)

    z = torch.randn(n_samples, 4, 16, 16, device=device)

    for i in tqdm.tqdm(range(len(step_indices)), desc=f'DDIM ({n_steps} steps)'):
        t = step_indices[i]
        t_prev = step_indices[i + 1] if i + 1 < len(step_indices) else 0

        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps_pred = model(z, t_batch)

        alpha_bar_t = schedule.alpha_bar[t]
        alpha_bar_prev = schedule.alpha_bar[t_prev] if t_prev > 0 else torch.tensor(1.0, device=device)

        # Predicted x_0
        x0_pred = (z - (1 - alpha_bar_t).sqrt() * eps_pred) / alpha_bar_t.sqrt()

        # DDIM deterministic update
        z = alpha_bar_prev.sqrt() * x0_pred + (1 - alpha_bar_prev).sqrt() * eps_pred

    images = vae.decode(z / scaling_factor).clamp(-1, 1)
    return images.cpu()

In [ ]:
# Сравнение DDIM при разном числе шагов
import time

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
step_counts = [1000, 100, 50, 10]

for ax, n_steps in zip(axes, step_counts):
    t0 = time.time()
    set_global_seed(42)
    imgs = sample_ddim_latent(model, schedule, vae, scaling_factor, n_samples=8, n_steps=n_steps)
    dt = time.time() - t0
    grid = make_grid(imgs, nrow=4, normalize=True, value_range=(-1, 1))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(f'DDIM {n_steps} steps\n({dt:.1f}s)')
    ax.axis('off')

fig.suptitle('DDIM: Quality vs Speed tradeoff', fontsize=14)
plt.tight_layout(); plt.show()

## Пошаговая визуализация денойзинга

Посмотрим, как изображение постепенно «проявляется» из шума:

In [ ]:
set_global_seed(7)
samples_vis, intermediates = sample_ddpm_latent(
    model, schedule, vae, scaling_factor, n_samples=4, return_intermediates=True
)

n_shown = len(intermediates) + 1
fig, axes = plt.subplots(4, n_shown, figsize=(n_shown * 2.5, 10))

t_labels = list(range(999, -1, -200)) + [0]

for row in range(4):
    for col, inter in enumerate(intermediates):
        img = inter[row].permute(1, 2, 0).numpy() * 0.5 + 0.5
        axes[row, col].imshow(np.clip(img, 0, 1))
        axes[row, col].axis('off')
        if row == 0:
            axes[0, col].set_title(f't ≈ {t_labels[col]}')

    img = samples_vis[row].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[row, -1].imshow(np.clip(img, 0, 1))
    axes[row, -1].axis('off')
    if row == 0:
        axes[0, -1].set_title('Final (t=0)')

fig.suptitle('Latent Diffusion: gradual denoising (decoded to pixels)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

##  Интерполяция в $z_T$

Сферическая интерполяция (slerp) между двумя шумами $z_T^{(1)}$ и $z_T^{(2)}$ — показывает гладкость генеративного пространства:

In [ ]:
def slerp(z1: torch.Tensor, z2: torch.Tensor, lam: float) -> torch.Tensor:
    """Spherical linear interpolation between two tensors.

    Args:
        z1: First tensor (flattened internally).
        z2: Second tensor (same shape as z1).
        lam: Interpolation parameter in [0, 1].

    Returns:
        Interpolated tensor of same shape.
    """
    z1_flat = z1.reshape(-1).float()
    z2_flat = z2.reshape(-1).float()
    cos_omega = (z1_flat @ z2_flat) / (z1_flat.norm() * z2_flat.norm() + 1e-8)
    cos_omega = cos_omega.clamp(-1, 1)
    omega = cos_omega.acos()
    if omega.abs() < 1e-6:
        return (1 - lam) * z1 + lam * z2
    s1 = (omega * (1 - lam)).sin() / omega.sin()
    s2 = (omega * lam).sin() / omega.sin()
    return (s1 * z1_flat + s2 * z2_flat).reshape(z1.shape)

In [ ]:
set_global_seed(42)
z1 = torch.randn(1, 4, 16, 16, device=device)
z2 = torch.randn(1, 4, 16, 16, device=device)

n_interp = 8
lambdas = np.linspace(0, 1, n_interp)
z_interps = torch.cat([slerp(z1, z2, l).unsqueeze(0) for l in lambdas], dim=0).to(device)

# Denoise all with DDIM
all_images = []
for i in range(n_interp):
    zi = z_interps[i:i+1]
    step_indices = sorted(set(np.linspace(0, schedule.T - 1, 50, dtype=int).tolist()), reverse=True)
    z = zi.clone()
    for j in range(len(step_indices)):
        t = step_indices[j]
        t_prev = step_indices[j + 1] if j + 1 < len(step_indices) else 0
        t_batch = torch.full((1,), t, device=device, dtype=torch.long)
        eps_pred = model(z, t_batch)
        ab_t = schedule.alpha_bar[t]
        ab_prev = schedule.alpha_bar[t_prev] if t_prev > 0 else torch.tensor(1.0, device=device)
        x0_pred = (z - (1 - ab_t).sqrt() * eps_pred) / ab_t.sqrt()
        z = ab_prev.sqrt() * x0_pred + (1 - ab_prev).sqrt() * eps_pred
    img = vae.decode(z / scaling_factor).clamp(-1, 1)
    all_images.append(img.cpu())

all_images = torch.cat(all_images, dim=0)
grid = make_grid(all_images, nrow=n_interp, normalize=True, value_range=(-1, 1))
fig, ax = plt.subplots(1, 1, figsize=(n_interp * 2.5, 3))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title('Slerp interpolation in z_T space (DDIM 50 steps)', fontsize=13)
ax.axis('off')
plt.tight_layout(); plt.show()

## Выводы

Ответьте на вопросы:

1. Как качество сэмплов DDIM зависит от числа шагов? Начиная с какого числа шагов качество приемлемо?
2. Сравните скорость генерации DDPM (1000 шагов) и DDIM (50 шагов). Во сколько раз DDIM быстрее?
3. Насколько плавная интерполяция в $z_T$ пространстве? Есть ли резкие переходы?
4. В чём преимущество обучения диффузии на латентах 16×16×4 по сравнению с пиксельным пространством 128×128×3?

# Итого

Мы реализовали полный пайплайн **Latent Diffusion Model**:

1. **VAE** (из предыдущего ДЗ): сжимает 128×128×3 → 16×16×4
2. **DDPM** в латентном пространстве: U-Net предсказывает шум на 16×16×4
3. **Сэмплинг**: DDPM (1000 шагов) и DDIM (произвольное число шагов)
4. **Декодирование**: VAE decoder превращает латент обратно в картинку

Это ровно та схема, которая используется в Stable Diffusion — только с меньшей моделью и без text conditioning.